## セル8：参照動画のモーション × 参照画像の人物

このノートの R2V は **ComfyUI の MiniMaxH3ReferenceToVideo（ref2va）** で動かす。

### うまく動かなかった原因（修正済み）

1. **セル8先頭が `print(= * 60)` で SyntaxError** → セル自体が実行できなかった
2. **動画配線に失敗すると `use_videos=False` へ黙ってフォールバック** → 人物画像だけの生成になり、モーション転写したように見えない
3. **セル3の既定 MODE が `t2v_i2v`** → R2V 必須の `ref2va` が入らない
4. **線画化（STRIP_VIDEO_IDENTITY=True）が既定ON** → H3 は実写クリップからモーションを読む。Canny 線画だと動きが消えることが多い
5. **LoadImage / VHS がファイル名だけ** → サブフォルダの素材を拾えない

### いまの正しい使い方

- セル3 の MODE は **both**（または r2v）
- セル8: IMAGE_FILES = 人物、VIDEO_FILES = 動き。`STRIP_VIDEO_IDENTITY` は顔が混ざるときだけ True
- 動画なしフォールバックはしない（失敗したらエラーで止める）


In [ ]:
#@title セル1：Google Drive 接続 + 準備チェック
# ##############################################################################
print("=" * 60)
print(" セル1：Google Drive 接続 + 準備")
print("=" * 60)

from google.colab import drive
import os

# ---------- 設定（必要なら変更）----------
# Drive 上の保存先（初回に自動作成）
DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"  #@param {type:"string"}
COMFY_DIR = "/content/ComfyUI"
# ----------------------------------------

drive.mount("/content/drive")

# Drive 側ディレクトリ
DRIVE_MODELS = f"{DRIVE_ROOT}/models"
for sub in ["diffusion_models", "text_encoders", "vae", "checkpoints", "loras", "controlnet"]:
    os.makedirs(f"{DRIVE_MODELS}/{sub}", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/input", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/workflows", exist_ok=True)

print("Drive 保存先:", DRIVE_ROOT)
print("モデル置き場:", DRIVE_MODELS)

# 容量表示
print("\n--- Colab (/content) ---")
!df -h /content | tail -1
print("--- Google Drive (目安) ---")
!df -h /content/drive 2>/dev/null | tail -1 || echo "(Drive の df は環境により出ないことがあります)"

!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    print("⚠ GPU OFF → ランタイム → ランタイムのタイプを変更 → GPU")

# 後続セル用にパスをファイルへ保存
with open("/content/h3_paths.env", "w") as f:
    f.write(f"DRIVE_ROOT={DRIVE_ROOT}\n")
    f.write(f"DRIVE_MODELS={DRIVE_MODELS}\n")
    f.write(f"COMFY_DIR={COMFY_DIR}\n")

print("\nOK → 次は【セル2】")


# ##############################################################################


In [ ]:
#@title セル2：ComfyUI インストール + Drive の models を接続
# ##############################################################################
print("=" * 60)
print(" セル2：ComfyUI + Drive リンク")
print("=" * 60)

import os

# パス読み込み
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_ROOT = env["DRIVE_ROOT"]
DRIVE_MODELS = env["DRIVE_MODELS"]
COMFY_DIR = env["COMFY_DIR"]

# ComfyUI は /content に（軽くて速い）
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git {COMFY_DIR}
else:
    print("ComfyUI 既存 → pull")
    %cd {COMFY_DIR}
    !git pull --ff-only || true

%cd {COMFY_DIR}
!pip install -q -r requirements.txt

# ---------------------------------------------------------------------------
# 速度用 custom_nodes（よく考えて入れた推奨セット）
#
# 判断メモ:
#   - 公式 H3 ドキュメントが推すのは Sage Attention ≒ 最大~2倍・品質ほぼ維持
#   - SolAttn_triton は sparse attention。H3 向けに使われ、+15〜20% の報告あり
#     ただし README も「experimental」、初回 Triton コンパイルで遅い、
#     品質は tau 次第で落ちる。作者テストは主に 4090/5090
#   - 4090 ベンチでは「H3 Mem Eff Sage」単独の方が Sol 単独より速い例が多い
#   - 結論: Sage を本命で必須寄りに入れ、Sol は ON で入れておくが
#            ワークフローでは「まず Sage だけ → 足りなければ Sol 追加」
# ---------------------------------------------------------------------------
INSTALL_SPEED_NODES = True  #@param {type:"boolean"}
# Sol-Attn リポジトリも clone する（ノードは使うときだけグラフに足す）
INSTALL_SOLATTN = True  #@param {type:"boolean"}
# SageAttention pip（失敗しても Comfy 自体は動く）
INSTALL_SAGEATTENTION = True  #@param {type:"boolean"}

cn = f"{COMFY_DIR}/custom_nodes"
os.makedirs(cn, exist_ok=True)

def clone_or_pull(url, folder_name):
    path = os.path.join(cn, folder_name)
    if os.path.isdir(path):
        print(f"更新: {folder_name}")
        !git -C "{path}" pull --ff-only || true
    else:
        print(f"clone: {folder_name}")
        !git clone --depth 1 "{url}" "{path}"
    req = os.path.join(path, "requirements.txt")
    if os.path.isfile(req):
        !pip install -q -r "{req}" || true
    return path

if INSTALL_SPEED_NODES:
    print("\n=== 速度ノード導入 ===")
    # 1) KJNodes = Patch Sage / H3 Mem Eff Sage など（本命）
    clone_or_pull("https://github.com/kijai/ComfyUI-KJNodes.git", "ComfyUI-KJNodes")

    # 2) Sol-Attn Triton（任意加速・実験的）
    if INSTALL_SOLATTN:
        clone_or_pull(
            "https://github.com/kijai/ComfyUI-SolAttn_triton.git",
            "ComfyUI-SolAttn_triton",
        )
        # Triton: Colab Linux では通常 torch 同梱 or pip で入る
        !pip install -q -U triton || true
        print("SolAttn: 初回生成は kernel コンパイルで遅くなることがあります")

    # 2.5) Video Helper Suite = VHS_LoadVideo（R2V モーション）
    INSTALL_VHS = True
    if INSTALL_VHS:
        clone_or_pull(
            "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
            "ComfyUI-VideoHelperSuite",
        )

    # 3) SageAttention 本体
    if INSTALL_SAGEATTENTION:
        print("SageAttention インストール試行...")
        # まず素直に pip。Colab の torch/cuda と合わない場合は失敗しても続行
        !pip install -q sageattention 2>/dev/null || \
         pip install -q git+https://github.com/thu-ml/SageAttention.git 2>/dev/null || \
         echo "SageAttention pip 失敗 → 後で wheel を合わせるか、KJNodes の一部機能のみで継続"

    try:
        import sageattention  # noqa: F401
        print("✓ sageattention import OK")
    except Exception as e:
        print("△ sageattention 未導入（標準 attention で動作。速度は落ちる）:", e)

    try:
        import triton  # noqa: F401
        print("✓ triton import OK:", getattr(triton, "__version__", "?"))
    except Exception as e:
        print("△ triton 未導入（SolAttn は動かない可能性）:", e)

    print("custom_nodes:")
    !ls -1 "{cn}"
else:
    print("速度ノードスキップ（INSTALL_SPEED_NODES=False）")

# --- models を Drive に接続（本体は Drive、Colab にはリンクだけ）---
def link_dir(link_path, target_path):
    """link_path → target_path のシンボリックリンク。既存は退避/削除して作り直す"""
    os.makedirs(target_path, exist_ok=True)
    if os.path.islink(link_path):
        os.remove(link_path)
    elif os.path.isdir(link_path):
        # 中身が空っぽい or リンクしたいのでリネーム退避
        bak = link_path + ".local_bak"
        if os.path.exists(bak):
            import shutil
            shutil.rmtree(bak, ignore_errors=True)
        os.rename(link_path, bak)
        print("  退避:", bak)
    elif os.path.exists(link_path):
        os.remove(link_path)
    os.symlink(target_path, link_path)
    print(f"  link: {link_path}  →  {target_path}")

print("\nDrive の models を ComfyUI に接続:")
models_root = f"{COMFY_DIR}/models"
os.makedirs(models_root, exist_ok=True)

# 主要サブフォルダを個別リンク（ComfyUI が他サブフォルダを作っても壊れにくい）
for sub in ["diffusion_models", "text_encoders", "vae", "checkpoints", "loras", "controlnet"]:
    link_dir(f"{models_root}/{sub}", f"{DRIVE_MODELS}/{sub}")

# output / input も Drive へ（生成物が消えない）
link_dir(f"{COMFY_DIR}/output", f"{DRIVE_ROOT}/output")
link_dir(f"{COMFY_DIR}/input", f"{DRIVE_ROOT}/input")

# 公式 workflow を Drive と user 両方へ
wf_user = f"{COMFY_DIR}/user/default/workflows"
os.makedirs(wf_user, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/workflows", exist_ok=True)
base = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates"
for name in [
    "video_minimax_h3_t2v.json",
    "video_minimax_h3_i2v.json",
    "video_minimax_h3_r2v.json",
]:
    !wget -q -O "{DRIVE_ROOT}/workflows/{name}" "{base}/{name}"
    !cp -f "{DRIVE_ROOT}/workflows/{name}" "{wf_user}/{name}"
    print("workflow:", name)

print("\n接続確認:")
!ls -la {COMFY_DIR}/models/diffusion_models | head -5
!ls -la {COMFY_DIR}/output | head -3

print("\nOK → 次は【セル3】（Drive に無ければDL、あればスキップ）")


# ##############################################################################



In [ ]:
#@title セル3：モデルを Google Drive へ取得（あればスキップ）
# ##############################################################################
print("=" * 60)
print(" セル3：モデル → Google Drive")
print("=" * 60)
print(" Colab のディスクではなく Drive に保存します")
print(" R2V には ref2va が必要 → 既定 MODE=both")
print("=" * 60)

import os

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_MODELS = env["DRIVE_MODELS"]
DRIVE_ROOT = env["DRIVE_ROOT"]

HF = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"

# モード: t2v_i2v だけなら約42GB / both だと +約21GB
MODE = "both"  #@param ["t2v_i2v", "r2v", "both"]
DIT_QUANT = "int8"  #@param ["int8", "fp8"]

if DIT_QUANT == "int8":
    fl2va = "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
    ref2va = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
else:
    fl2va = "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"
    ref2va = "minimax_h3_ref2va_pruned_fp8_scaled.safetensors"

files = [
    (f"text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
     f"{DRIVE_MODELS}/text_encoders"),
    (f"vae/minimax_h3_video_vae_fp16.safetensors",
     f"{DRIVE_MODELS}/vae"),
    (f"vae/minimax_h3_audio_vae_fp32.safetensors",
     f"{DRIVE_MODELS}/vae"),
]
if MODE in ("t2v_i2v", "both"):
    files.append((f"diffusion_models/{fl2va}", f"{DRIVE_MODELS}/diffusion_models"))
if MODE in ("r2v", "both"):
    files.append((f"diffusion_models/{ref2va}", f"{DRIVE_MODELS}/diffusion_models"))

MIN_OK = 1_000_000  # 1MB 未満は失敗扱い

for rel, folder in files:
    os.makedirs(folder, exist_ok=True)
    name = os.path.basename(rel)
    path = os.path.join(folder, name)
    if os.path.exists(path) and os.path.getsize(path) > MIN_OK:
        gb = os.path.getsize(path) / 1e9
        print(f"✓ Drive にあり（スキップ）: {name}  ({gb:.2f} GB)")
        continue
    url = f"{HF}/{rel}"
    print(f"↓ Drive へダウンロード: {name}")
    print(f"  保存先: {path}")
    # -c で再開可能（途中切断に強い）
    !wget -c --show-progress -O "{path}" "{url}"
    if not os.path.exists(path) or os.path.getsize(path) < MIN_OK:
        # 壊れたファイルを残さない
        if os.path.exists(path):
            os.remove(path)
        raise SystemExit(f"DL 失敗: {name}\nDrive の空き容量を確認してください（目安 45GB+）")
    print(f"✓ 完了: {name}  ({os.path.getsize(path)/1e9:.2f} GB)")

print("\n--- Drive 上の MiniMax 関連ファイル ---")
!find "{DRIVE_MODELS}" -type f \( -name "*minimax*" -o -name "*qwen3vl*" \) -printf "%s\t%p\n" 2>/dev/null | awk '{printf "%.2f GB\t%s\n", $1/1e9, $2}'

print("\nColab ディスク:")
!df -h /content | tail -1
print("\nOK → 次は【セル4】起動")


# ##############################################################################


In [ ]:
#@title セル4：起動して UI を開く（loca.lt / colab.dev は使わない）
# ##############################################################################
print("=" * 60)
print(" セル4：ComfyUI 起動")
print("=" * 60)
print(" 開くのは loca.lt の URL だけ（colab.dev は開かない）")
print("=" * 60)

import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
COMFY_DIR = env["COMFY_DIR"]
DRIVE_ROOT = env["DRIVE_ROOT"]
DRIVE_MODELS = env["DRIVE_MODELS"]

PORT = 8188
LOG = "/content/comfyui.log"
# A100 など余裕あり: --highvram / 足りない: --lowvram
EXTRA = "--lowvram"  #@param ["--highvram", "--normalvram", "--lowvram", "--novram"]

if not os.path.isfile(f"{COMFY_DIR}/main.py"):
    raise SystemExit("セル2が未実行です")

# リンクが切れていたら付け直す
def ensure_link(link_path, target_path):
    os.makedirs(target_path, exist_ok=True)
    if os.path.islink(link_path) and os.readlink(link_path) == target_path:
        return
    if os.path.islink(link_path) or os.path.exists(link_path):
        if os.path.islink(link_path):
            os.remove(link_path)
        elif os.path.isdir(link_path) and not os.path.islink(link_path):
            # 既に中身がある場合は触らない（Drive 接続済み想定）
            if os.listdir(link_path):
                return
            os.rmdir(link_path)
        else:
            os.remove(link_path)
    if not os.path.exists(link_path):
        os.symlink(target_path, link_path)

for sub in ["diffusion_models", "text_encoders", "vae"]:
    ensure_link(f"{COMFY_DIR}/models/{sub}", f"{DRIVE_MODELS}/{sub}")
ensure_link(f"{COMFY_DIR}/output", f"{DRIVE_ROOT}/output")

# モデル存在チェック
need = [
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    f"{DRIVE_MODELS}/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    f"{DRIVE_MODELS}/vae/minimax_h3_video_vae_fp16.safetensors",
    f"{DRIVE_MODELS}/vae/minimax_h3_audio_vae_fp32.safetensors",
]
# fp8 の場合も許容
alts = [
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors",
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
]
ref2va_ok = any(
    os.path.exists(p) and os.path.getsize(p) > 1_000_000
    for p in [
        f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
        f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
    ]
)
if not ref2va_ok:
    print("⚠ ref2va が無いとセル8の R2V は動きません。セル3 MODE=both かセル5 を実行してください。")
missing = [p for p in need if not (os.path.exists(p) and os.path.getsize(p) > 1_000_000)]
if missing and not any(os.path.exists(a) and os.path.getsize(a) > 1_000_000 for a in alts):
    # fl2va だけ alt 可、TE/VAE は必須
    te_vae_miss = [p for p in need[1:] if not (os.path.exists(p) and os.path.getsize(p) > 1_000_000)]
    fl_ok = (os.path.exists(need[0]) and os.path.getsize(need[0]) > 1_000_000) or any(
        os.path.exists(a) and os.path.getsize(a) > 1_000_000 for a in alts
    )
    if te_vae_miss or not fl_ok:
        print("モデル不足。セル3を実行してください:")
        for p in missing:
            print(" -", p)
        raise SystemExit(1)

print("モデル確認 OK（Drive）")
!ls -lh "{DRIVE_MODELS}/diffusion_models" | head -10

for c in [
    f"fuser -k {PORT}/tcp",
    "pkill -f 'python.*main.py'",
    "pkill -f localtunnel",
    "pkill -f cloudflared",
]:
    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

os.chdir(COMFY_DIR)
log_f = open(LOG, "w", buffering=1)
cmd = [
    sys.executable, "main.py",
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    EXTRA,
    "--disable-auto-launch",
    "--enable-cors-header",
]
print("starting:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True)
print("PID:", proc.pid)

print("起動待ち...")
ok = False
for i in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
        ok = True
        break
    except Exception:
        if proc.poll() is not None:
            break
        time.sleep(2)

if not ok:
    print(open(LOG, errors="replace").read()[-4000:])
    raise SystemExit("起動失敗")

print("✓ ComfyUI 起動OK")

# iframe
try:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_iframe(PORT, height=900)
    print("【方法A】下の埋め込み UI を使ってもOK")
except Exception as e:
    print("iframe:", e)

# localtunnel
if shutil.which("npm") is None:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1
    !apt-get install -y -qq nodejs >/dev/null 2>&1

try:
    password = urllib.request.urlopen(
        "https://loca.lt/mytunnelpassword", timeout=15
    ).read().decode().strip()
except Exception:
    password = !curl -s https://loca.lt/mytunnelpassword
    password = password[0] if password else "?"

lt_log = "/content/localtunnel.log"
lt_f = open(lt_log, "w", buffering=1)
subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", str(PORT)],
    stdout=lt_f, stderr=subprocess.STDOUT, start_new_session=True,
)

url = None
for _ in range(50):
    time.sleep(1)
    try:
        text = open(lt_log, errors="replace").read()
    except Exception:
        text = ""
    m = re.search(r"https://[a-z0-9.-]+\.loca\.lt", text, re.I)
    if m:
        url = m.group(0)
        break

print()
print("#" * 64)
print("#  開く URL はこれだけ（loca.lt）")
if url:
    print("# ", url)
    print("#  パスワード:", password)
else:
    print("#  自動取得失敗 → !npx --yes localtunnel --port 8188")
print("#  × colab.dev / trycloudflare は開かない")
print("#  生成結果 Drive:", f"{DRIVE_ROOT}/output")
print("#" * 64)

try:
    from IPython.display import display, HTML
    if url:
        display(HTML(f"""
        <div style="padding:16px;border:3px solid #0a0;background:#e8ffe8;font-size:18px">
          <b>ComfyUI を開く</b><br><br>
          <a href="{url}" target="_blank" style="font-size:22px">{url}</a><br><br>
          パスワード: <code style="font-size:20px">{password}</code><br><br>
          モデル・出力は Google Drive:<br>
          <code>{DRIVE_ROOT}</code><br><br>
          <span style="color:#a00">colab.dev は開かない</span>
        </div>
        """))
except Exception:
    pass

print("ログ:", LOG)
print("Drive output:", f"{DRIVE_ROOT}/output")


# ##############################################################################


In [ ]:
#@title セル5：（任意）R2V モデルを Drive に追加
# ##############################################################################
print("参照モード(R2V)が必要なときだけ")

import os
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
path = f"{env['DRIVE_MODELS']}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"
url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"
if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
    print("✓ すでに Drive にあります:", path)
else:
    print("↓ Drive へ DL:", path)
    !wget -c --show-progress -O "{path}" "{url}"
    print("完了")


# ##############################################################################


In [ ]:
#@title セル6：（任意）Drive 上のモデル一覧・容量
# ##############################################################################
import os
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
root = env["DRIVE_ROOT"]
print("Drive root:", root)
!du -sh "{root}" 2>/dev/null || true
!du -sh "{root}/models"/* 2>/dev/null || true
!find "{root}/models" -type f -printf "%s %p\n" 2>/dev/null | sort -n | tail -20


In [ ]:
#@title セル5b：LightX2V Turbo 4-step v1.0 LoRA を Drive に追加（高速化）
# ##############################################################################
print("=" * 60)
print(" セル5b：lightx2v turbo 4step v1.0 → models/loras")
print("=" * 60)

import os
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v

DRIVE_MODELS = env["DRIVE_MODELS"]
lora_dir = f"{DRIVE_MODELS}/loras"
os.makedirs(lora_dir, exist_ok=True)

LORA_NAME = "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"
LORA_URL = "https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"
path = f"{lora_dir}/{LORA_NAME}"

if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
    print("既にあります（スキップ）:")
    print(" ", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
else:
    print("ダウンロード中…")
    print(LORA_URL)
    !wget -c --show-progress -O "{path}" "{LORA_URL}"
    if not (os.path.exists(path) and os.path.getsize(path) > 1_000_000):
        raise SystemExit("LoRA DL 失敗。URL とネットを確認してください")
    print("保存:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")

comfy_lora = Path(env["COMFY_DIR"]) / "models" / "loras" / LORA_NAME
print("ComfyUI loras:", comfy_lora, "exists=", comfy_lora.exists())
print("次: セル8 で USE_LORA=True / STEPS=4。R2V は Ref2V turbo を使う（FL2V turbo ではない）")
!ls -lh "{lora_dir}" | tail -20



# Also fetch Ref2V turbo (required for R2V speed + quality; FL2V turbo is wrong model family)
REF2V_LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
REF2V_URL = "https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
_ref_path = f"{lora_dir}/{REF2V_LORA}"
if os.path.exists(_ref_path) and os.path.getsize(_ref_path) > 1_000_000:
    print("Ref2V turbo already present:", _ref_path)
else:
    print("Downloading Ref2V turbo...")
    import urllib.request
    urllib.request.urlretrieve(REF2V_URL, _ref_path)
    print("saved", _ref_path, os.path.getsize(_ref_path) if os.path.exists(_ref_path) else 0)


In [ ]:
#@title セル7：input 内の参照画像・動画を一覧（R2V 素材確認）
# ##############################################################################
print("=" * 60)
print(" セル7：参照素材一覧（Drive input）")
print("=" * 60)

import os
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_ROOT = env["DRIVE_ROOT"]
COMFY_DIR = env["COMFY_DIR"]
INP = Path(COMFY_DIR) / "input"
os.makedirs(INP, exist_ok=True)

IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VID_EXT = {".mp4", ".mov", ".webm", ".mkv", ".avi"}

images = sorted(
    [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT],
    key=lambda p: p.name.lower(),
)
videos = sorted(
    [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in VID_EXT],
    key=lambda p: p.name.lower(),
)

print(f"input dir: {INP}")
print(f"images: {len(images)}  (R2V max 9)")
for i, p in enumerate(images, 1):
    print(f"  [{i:02d}] {p.relative_to(INP)}  ({p.stat().st_size/1e6:.2f} MB)")
print(f"videos: {len(videos)}  (R2V motion max 3)")
for i, p in enumerate(videos, 1):
    print(f"  [{i:02d}] {p.relative_to(INP)}  ({p.stat().st_size/1e6:.2f} MB)")

if not images and not videos:
    print("\n素材なし。Drive に置いてください:")
    print(f"  {DRIVE_ROOT}/input/")
else:
    print("\n次は【セル8】で IMAGE_FILES / VIDEO_FILES を指定して R2V")



In [ ]:
%%writefile h3_r2v_core.py
"""Helpers for MiniMax H3 ComfyUI R2V (identity from stills, motion from video).

No ComfyUI / network required. Used by minimax_h3_colab_完全版.ipynb cell 8.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any


def frames(duration_s: float) -> int:
    """H3 length grid: 17k+5 at 24fps."""
    base = max(5, int(round(float(duration_s) * 24)))
    return int(base + (5 - (base % 17)) % 17)


def parse_list(s: str) -> list[str]:
    s = (s or "").strip()
    if not s or s.lower() in ("none", "-", "null"):
        return []
    return [x.strip().lstrip("./") for x in s.replace(";", ",").split(",") if x.strip()]


def comfy_media_name(rel: str | Path) -> str:
    """ComfyUI LoadImage / VHS video widgets want a path relative to input/, not basename-only."""
    return str(rel).replace("\\", "/").lstrip("./")


def prefer_ref2v_lora(lora_paths: list[Path], use_lora: bool) -> str | None:
    """Prefer Ref2V turbo for ref2va unet; never prefer FL2V-only when ref2v exists."""
    if not use_lora or not lora_paths:
        return None
    names = [p.name for p in lora_paths if p.suffix.lower() == ".safetensors"]
    if not names:
        return None

    def score(n: str) -> tuple:
        nl = n.lower()
        is_ref2v = 1 if ("ref2v" in nl or "ref2va" in nl) else 0
        is_fl2v = 1 if ("fl2v" in nl or "fl2va" in nl) and not is_ref2v else 0
        is_turbo = 1 if "turbo" in nl else 0
        is_4step = 1 if "4step" in nl or "4_step" in nl else 0
        is_comfy = 1 if "comfyui" in nl else 0
        return (is_ref2v, is_turbo, is_4step, is_comfy, -is_fl2v, n)

    ref2v = [n for n in names if "ref2v" in n.lower() or "ref2va" in n.lower()]
    if ref2v:
        return sorted(ref2v, key=score, reverse=True)[0]
    return sorted(names, key=score, reverse=True)[0]


def role_lock_preamble(img_names: list[str], vid_names: list[str]) -> str:
    lines = [
        "ROLE LOCK (mandatory):",
        "REFERENCE VIDEO = MOTION ONLY (hard rule):",
        "- From every <Video N>, read ONLY: body motion, hand trajectories, footwork, camera path,",
        "  framing changes, pacing, cut rhythm, and timing.",
        "- From every <Video N>, IGNORE completely: faces, gender presentation, age, hair, skin tone,",
        "  body build, costumes, logos, on-screen text, and the original actor's identity.",
        "- Never retarget appearance from the motion clip. The motion clip is a choreography/camera guide only.",
    ]
    for i, n in enumerate(img_names):
        lines.append(
            f"- <Picture {i+1}> ({n}) = IDENTITY + COSTUME only. "
            "Copy exact face, hair, skin, body proportions, and wardrobe from this still. "
            "Do NOT replace this person with anyone visible in any motion video."
        )
    for i, n in enumerate(vid_names):
        lines.append(
            f"- <Video {i+1}> ({n}) = MOTION + CAMERA + TIMING ONLY (not appearance). "
            "Follow camera path, pacing, body action, shot rhythm. "
            "Do NOT invent different choreography. "
            "Do NOT copy faces, body type, age, hair, or costumes from people in this video. "
            "Drive the still-locked characters through this motion like motion capture."
        )
    if not vid_names:
        lines.append("- No motion video: invent plausible cinematic motion consistent with the stills.")
    lines.append(
        "CONFLICT RULE: appearance always wins from Pictures; motion always wins from Videos. "
        "If the video actors look different from the stills, keep the still faces and only transfer motion."
    )
    return "\n".join(lines) + "\n\n"


def build_default_prompt(img_names: list[str], vid_names: list[str], duration_s: float) -> str:
    lock = role_lock_preamble(img_names, vid_names)
    pic_lines = [f"- <Picture {i+1}> = {n}" for i, n in enumerate(img_names)]
    vid_lines = [
        f"- <Video {i+1}> = {n} (MOTION ONLY: body/camera/timing — ignore faces & costumes in this clip)"
        for i, n in enumerate(vid_names)
    ]
    extra = ""
    if vid_names:
        extra = (
            " Transfer ONLY motion and camera from <Video 1> onto those still-locked characters "
            "(motion-capture style). Do not inherit the video actors' appearance."
        )
    pics = " and <Picture 2>" if len(img_names) > 1 else ""
    return (
        lock
        + "Use these references:\n"
        + "\n".join(pic_lines + vid_lines)
        + "\nSTYLE: photorealistic live-action cinematic, real skin pores, natural materials, "
        "no anime cel, no text, no subtitles, no logos.\n"
        f"integrated_multimodal_description: [Shot 1] Live-action remake for about "
        f"{duration_s:.0f} seconds. Characters must match <Picture 1>{pics} faces exactly."
        + extra
        + "\noverall_soundscape: Ambient and action SFX matching motion."
        + "\nnon_diegetic_music: None."
    )


def finalize_prompt(
    prompt: str,
    img_names: list[str],
    vid_names: list[str],
    duration_s: float,
    inject_role_lock: bool = True,
) -> str:
    raw = (prompt or "").strip()
    if not raw:
        return build_default_prompt(img_names, vid_names, duration_s)
    pics = "\n".join(f"<Picture {i+1}>:{n}" for i, n in enumerate(img_names))
    vids = "\n".join(f"<Video {i+1}>:{n}" for i, n in enumerate(vid_names))
    out = raw.replace("{pictures}", pics).replace("{videos}", vids)
    if inject_role_lock:
        has_lock = "ROLE LOCK" in out or "MOTION LOCK" in out or "APPEARANCE LOCK" in out
        if not has_lock:
            out = role_lock_preamble(img_names, vid_names) + out
        if img_names and "<Picture 1>" not in out and "<picture 1>" not in out.lower():
            out = f"Identity for character 1 is locked to <Picture 1> ({img_names[0]}).\n" + out
        if len(img_names) > 1 and "<Picture 2>" not in out:
            out = f"Identity for character 2 is locked to <Picture 2> ({img_names[1]}).\n" + out
        if vid_names and "<Video 1>" not in out:
            out = (
                f"MOTION ONLY from <Video 1> ({vid_names[0]}): body action, camera, timing. "
                "Ignore faces/costumes in the video; keep still-image identity.\n"
                + out
            )
        if vid_names and "MOTION ONLY" not in out.upper() and "motion only" not in out.lower():
            out = (
                "HARD CONSTRAINT: Reference videos provide MOTION ONLY "
                "(choreography + camera + timing). Appearance comes exclusively from still Pictures.\n"
                + out
            )
    return out


def image_ref_key(i: int) -> str:
    return f"ref_images.ref_image_{i}"


def video_ref_key(i: int) -> str:
    return f"ref_videos.ref_video_{i}"


def vhs_load_video_inputs(
    object_info: dict[str, Any] | None,
    filename: str,
    length: int,
) -> dict[str, Any]:
    """Fill VHS_LoadVideo widgets from live object_info so schema drift does not drop the clip."""
    filename = comfy_media_name(filename)
    info = ((object_info or {}).get("VHS_LoadVideo") or {}).get("input") or {}
    required = info.get("required") or {}
    optional = info.get("optional") or {}
    merged = {**required, **optional}
    inputs: dict[str, Any] = {}
    for name, spec in merged.items():
        if isinstance(spec, list) and len(spec) > 1 and isinstance(spec[1], dict) and "default" in spec[1]:
            inputs[name] = spec[1]["default"]
    if "video" in merged:
        inputs["video"] = filename
    elif "file" in merged:
        inputs["file"] = filename
    else:
        inputs["video"] = filename
    if "force_rate" in merged:
        inputs["force_rate"] = 24
    if "frame_load_cap" in merged:
        inputs["frame_load_cap"] = int(length)
    if "skip_first_frames" in merged:
        inputs["skip_first_frames"] = 0
    if "select_every_nth" in merged:
        inputs["select_every_nth"] = 1
    if "force_size" in merged:
        inputs["force_size"] = "Disabled"
    return inputs


def native_load_video_inputs(filename: str) -> dict[str, Any]:
    name = comfy_media_name(filename)
    return {"file": name, "video": name}


def build_r2v_graph(
    *,
    img_names: list[str],
    vid_names: list[str],
    prompt: str,
    unet: str,
    lora_name: str | None,
    lora_strength: float,
    width: int,
    height: int,
    duration_s: float,
    seed: int,
    steps: int,
    filename_prefix: str,
    ref_image_size: str = "max",
    use_videos: bool = True,
    has_vhs: bool = True,
    has_lora_loader: bool = True,
    has_audio_decode: bool = True,
    object_info: dict[str, Any] | None = None,
    clip_name: str = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    vvae: str = "minimax_h3_video_vae_fp16.safetensors",
    avae: str = "minimax_h3_audio_vae_fp32.safetensors",
) -> dict[str, Any]:
    """Build ComfyUI API graph for MiniMaxH3ReferenceToVideo."""
    if width % 32 or height % 32:
        raise ValueError(f"H3 width/height must be multiples of 32, got {width}x{height}")
    g: dict[str, Any] = {}
    length = frames(duration_s)

    for i, fname in enumerate(img_names):
        g[str(100 + i)] = {
            "class_type": "LoadImage",
            "inputs": {"image": comfy_media_name(fname)},
        }

    g["1"] = {
        "class_type": "UNETLoader",
        "inputs": {"unet_name": unet, "weight_dtype": "default"},
    }
    model: list[Any] = ["1", 0]
    if lora_name and has_lora_loader:
        g["2"] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {
                "model": ["1", 0],
                "lora_name": lora_name,
                "strength_model": float(lora_strength),
            },
        }
        model = ["2", 0]

    g["3"] = {
        "class_type": "CLIPLoader",
        "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"},
    }
    g["4"] = {"class_type": "VAELoader", "inputs": {"vae_name": vvae}}
    g["5"] = {"class_type": "VAELoader", "inputs": {"vae_name": avae}}

    r_inputs: dict[str, Any] = {
        "clip": ["3", 0],
        "vae": ["4", 0],
        "audio_vae": ["5", 0],
        "prompt": prompt,
        "width": int(width),
        "height": int(height),
        "length": length,
        "ref_image_size": ref_image_size if ref_image_size in ("match", "max") else "max",
    }

    for i in range(len(img_names)):
        r_inputs[image_ref_key(i)] = [str(100 + i), 0]

    if use_videos and vid_names:
        for vi, vname in enumerate(vid_names[:3]):
            node_id = str(190 + vi)
            if has_vhs:
                g[node_id] = {
                    "class_type": "VHS_LoadVideo",
                    "inputs": vhs_load_video_inputs(object_info, vname, length),
                }
            else:
                g[node_id] = {
                    "class_type": "LoadVideo",
                    "inputs": native_load_video_inputs(vname),
                }
            r_inputs[video_ref_key(vi)] = [node_id, 0]

    for bad in ("ref_videos", "ref_images", "ref_audios", "ref_video_audios"):
        r_inputs.pop(bad, None)

    g["20"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": r_inputs}
    g["21"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}}
    sampler = "euler" if lora_name else "res_multistep"
    scheduler = "simple" if lora_name else "beta"
    g["22"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler}}
    g["23"] = {
        "class_type": "BasicScheduler",
        "inputs": {
            "model": model,
            "scheduler": scheduler,
            "steps": int(steps) if lora_name else max(int(steps), 16),
            "denoise": 1.0,
        },
    }
    g["24"] = {
        "class_type": "BasicGuider",
        "inputs": {"model": model, "conditioning": ["20", 0]},
    }
    g["25"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["21", 0],
            "guider": ["24", 0],
            "sampler": ["22", 0],
            "sigmas": ["23", 0],
            "latent_image": ["20", 1],
        },
    }
    g["26"] = {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["25", 0], "vae": ["4", 0]},
    }
    if has_audio_decode:
        g["27"] = {
            "class_type": "VAEDecodeAudio",
            "inputs": {"samples": ["25", 0], "vae": ["5", 0]},
        }
        g["28"] = {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "audio": ["27", 0], "fps": 24},
        }
    else:
        g["28"] = {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "fps": 24},
        }
    g["29"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["28", 0],
            "filename_prefix": filename_prefix,
            "format": "auto",
            "codec": "auto",
        },
    }
    return g


def assert_graph_identity_motion(
    graph: dict[str, Any],
    *,
    expect_images: int,
    expect_videos: int,
    prompt: str,
) -> list[str]:
    """Return list of error strings (empty = ok)."""
    errs: list[str] = []
    node = graph.get("20") or {}
    if node.get("class_type") != "MiniMaxH3ReferenceToVideo":
        errs.append("node 20 is not MiniMaxH3ReferenceToVideo")
        return errs
    inputs = node.get("inputs") or {}
    for bad in ("ref_videos", "ref_images"):
        if bad in inputs:
            errs.append(f"parent key {bad} must not be used alone")
    for i in range(expect_images):
        k = image_ref_key(i)
        if k not in inputs:
            errs.append(f"missing {k}")
    if expect_videos:
        for i in range(expect_videos):
            k = video_ref_key(i)
            if k not in inputs:
                errs.append(f"missing {k}")
        if "190" not in graph:
            errs.append("missing video load node 190")
        else:
            vin = graph["190"].get("inputs") or {}
            klass = graph["190"].get("class_type")
            if klass == "VHS_LoadVideo" and int(vin.get("force_rate") or 0) not in (0, 24):
                # 0 = keep source rate in some VHS versions; 24 is required by H3
                errs.append("VHS force_rate must be 24")
            media = vin.get("video") or vin.get("file") or ""
            if not media:
                errs.append("video loader has empty filename")
    ris = inputs.get("ref_image_size")
    if ris not in ("match", "max"):
        errs.append(f"bad ref_image_size: {ris}")
    if expect_images and "<Picture 1>" not in prompt and "Picture 1" not in prompt:
        errs.append("prompt missing Picture 1 identity lock")
    if expect_videos and "<Video 1>" not in prompt and "Video 1" not in prompt:
        errs.append("prompt missing Video 1 motion lock")
    if expect_videos:
        pu = prompt.upper()
        if "MOTION ONLY" not in pu and "MOTION + CAMERA" not in pu and "MOTION/CAMERA" not in pu:
            if "MOTION" not in pu:
                errs.append("prompt missing MOTION-only language for reference video")
        if "FACE" not in pu and "IDENTITY" not in pu and "PICTURE" not in pu:
            errs.append("prompt missing still-identity vs video-motion separation")
    return errs


def _find_ffmpeg() -> str | None:
    import shutil

    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    for c in ("/usr/bin/ffmpeg", "/usr/local/bin/ffmpeg"):
        if Path(c).is_file():
            return c
    return None


def strip_motion_identity_video(
    src: Path,
    dst: Path,
    *,
    mode: str = "edges",
    ffmpeg_bin: str | None = None,
) -> Path:
    """Optional last resort: destroy photoreal identity while keeping coarse motion."""
    import subprocess

    src = Path(src)
    dst = Path(dst)
    if not src.is_file():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    mode = (mode or "edges").lower().strip()
    if mode not in ("edges", "blur"):
        mode = "edges"
    ff = ffmpeg_bin or _find_ffmpeg()
    if not ff:
        raise RuntimeError("Cannot strip video identity: ffmpeg is required")
    if mode == "blur":
        vf = "format=gray,gblur=sigma=12,format=yuv420p"
    else:
        vf = "format=gray,edgedetect=mode=colormix:high=0.12:low=0.04,format=yuv420p"
    r = subprocess.run(
        [ff, "-y", "-i", str(src), "-vf", vf, "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(dst)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not dst.is_file() or dst.stat().st_size < 1000:
        raise RuntimeError(f"ffmpeg strip failed: {(r.stderr or '')[-800:]}")
    return dst


def prepare_motion_refs(
    inp_dir: Path,
    rel_names: list[str],
    *,
    enabled: bool = True,
    mode: str = "edges",
) -> list[str]:
    if not enabled or not rel_names:
        return list(rel_names)
    out_names: list[str] = []
    for rel in rel_names:
        src = Path(inp_dir) / rel
        if not src.is_file():
            hits = list(Path(inp_dir).rglob(Path(rel).name))
            if not hits:
                raise FileNotFoundError(rel)
            src = hits[0]
        stem = src.stem
        if stem.endswith("_motion_only_edges") or stem.endswith("_motion_only_blur"):
            try:
                out_names.append(str(src.relative_to(inp_dir)).replace("\\", "/"))
            except ValueError:
                out_names.append(src.name)
            continue
        suffix = "_motion_only_edges" if mode == "edges" else "_motion_only_blur"
        dst = src.with_name(f"{stem}{suffix}.mp4")
        strip_motion_identity_video(src, dst, mode=mode)
        try:
            out_names.append(str(dst.relative_to(inp_dir)).replace("\\", "/"))
        except ValueError:
            out_names.append(dst.name)
    return out_names


In [ ]:
#@title セル8a：h3_r2v_core.py をデスクトップ作業フォルダへ上書き
from pathlib import Path

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
src = Path("h3_r2v_core.py")
if not src.is_file():
    src = Path("/content/h3_r2v_core.py")
if not src.is_file():
    raise SystemExit("h3_r2v_core.py がありません。直前の writefile セルを実行してください。")

body = src.read_text(encoding="utf-8")
dests = [
    DESKTOP / "h3_r2v_core.py",
    Path("/content/h3_r2v_core.py"),
    Path("/content/drive/MyDrive/minimax-h3-comfyui/h3_r2v_core.py"),
    Path.cwd() / "h3_r2v_core.py",
]
seen = set()
for d in dests:
    try:
        key = str(d.resolve()) if d.exists() or d.parent.exists() else str(d)
        if key in seen:
            continue
        seen.add(key)
        if d == DESKTOP / "h3_r2v_core.py" and not DESKTOP.is_dir():
            print("skip (no Windows folder):", d)
            continue
        d.parent.mkdir(parents=True, exist_ok=True)
        d.write_text(body, encoding="utf-8")
        print("overwrote", d)
    except Exception as e:
        print("skip", d, ":", e)
print("Desktop folder exists:", DESKTOP.is_dir())


In [ ]:
#@title セル8：R2V（参照画像=人物 / 参照動画=モーション）
# ##############################################################################
print("=" * 60)
print(" セル8：R2V — stills=identity, video=motion")
print("=" * 60)

import json, os, sys, time, uuid, urllib.request, urllib.error, shutil
from pathlib import Path

# ---------- ユーザー設定 ----------
IMAGE_FILES = "Image 1.jpg,Image 2.jpg"  #@param {type:"string"}
VIDEO_FILES = "0815(1).mp4"  #@param {type:"string"}
PROMPT = ""  #@param {type:"string"}
WIDTH = 960  #@param {type:"integer"}
HEIGHT = 544  #@param {type:"integer"}
DURATION_S = 6  #@param {type:"number"}
STEPS = 4  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
USE_LORA = True  #@param {type:"boolean"}
LORA_STRENGTH = 1.0  #@param {type:"number"}
REF_IMAGE_SIZE = "max"  #@param ["max", "match"]
FILENAME_PREFIX = "video/h3_r2v_flex"  #@param {type:"string"}
USE_MOTION = True  #@param {type:"boolean"}
INJECT_ROLE_LOCK = True  #@param {type:"boolean"}
# 既定OFF。線画化すると H3 がモーションを読めなくなることが多い。顔が混ざるときだけ True。
STRIP_VIDEO_IDENTITY = False  #@param {type:"boolean"}
STRIP_MODE = "edges"  #@param ["edges", "blur"]
DRY_RUN = False  #@param {type:"boolean"}
# --------------------------------

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
for p in (DESKTOP, Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/minimax-h3-comfyui")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from h3_r2v_core import (
    frames, parse_list, prefer_ref2v_lora, finalize_prompt, build_r2v_graph,
    image_ref_key, video_ref_key, assert_graph_identity_motion, prepare_motion_refs,
    comfy_media_name,
)

env = {}
env_path = Path("/content/h3_paths.env")
if env_path.is_file():
    with open(env_path) as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                env[k] = v
else:
    env = {"COMFY_DIR": str(Path.cwd() / "ComfyUI"), "DRIVE_ROOT": str(Path.cwd())}

COMFY_DIR = Path(env.get("COMFY_DIR", "/content/ComfyUI"))
DRIVE_ROOT = env.get("DRIVE_ROOT", "/content/drive/MyDrive/minimax-h3-comfyui")
INP = COMFY_DIR / "input"
PORT = 8188
MAX_IMAGES, MAX_VIDEOS = 9, 3
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VID_EXT = {".mp4", ".mov", ".webm", ".mkv", ".avi"}

def list_media(exts):
    if not INP.exists():
        return []
    return sorted(
        [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in exts],
        key=lambda p: str(p.relative_to(INP)).lower(),
    )

def resolve_one(n, kind, known_paths):
    n = (n or "").strip().strip('"').strip("'").lstrip("./")
    if not n:
        raise SystemExit(f"empty {kind} name")
    candidates = [INP / n, INP / Path(n).name]
    stem = Path(n).stem if Path(n).suffix else n
    exts = list(VID_EXT) if kind == "video" else list(IMG_EXT)
    if not Path(n).suffix:
        for e in exts:
            candidates += [INP / f"{n}{e}", INP / f"{stem}{e}"]
    for c in candidates:
        if c.is_file():
            return c
    hits = list(INP.rglob(Path(n).name)) if INP.exists() else []
    if hits:
        return hits[0]
    for p in known_paths:
        if p.stem == stem or p.name == n:
            return p
    available = ", ".join(p.name for p in known_paths[:40]) or "(none)"
    raise SystemExit(f"{kind} not found: {n}\ninput: {INP}\navailable: {available}")

def resolve_names(user_list, auto_paths, limit, kind):
    if user_list is not None and len(user_list) == 0 and kind == "video" and not USE_MOTION:
        return []
    if user_list:
        names = []
        for n in user_list[:limit]:
            p = resolve_one(n, kind, auto_paths)
            try:
                rel = str(p.relative_to(INP)).replace("\\", "/")
            except ValueError:
                rel = p.name
            print(f"  resolved {kind}: {n!r} -> {rel}")
            names.append(rel)
        return names
    return [str(p.relative_to(INP)).replace("\\", "/") for p in auto_paths[:limit]]

all_imgs = list_media(IMG_EXT)
all_vids = list_media(VID_EXT)
img_names = (
    resolve_names(None, all_imgs, MAX_IMAGES, "image")
    if IMAGE_FILES.strip() == ""
    else resolve_names(parse_list(IMAGE_FILES), all_imgs, MAX_IMAGES, "image")
)

if not USE_MOTION or VIDEO_FILES.strip().lower() in ("none", "-"):
    vid_names = []
elif VIDEO_FILES.strip() == "":
    vid_names = resolve_names(None, all_vids, MAX_VIDEOS, "video")
else:
    vid_names = resolve_names(parse_list(VIDEO_FILES), all_vids, MAX_VIDEOS, "video")

print("selected images:", img_names)
print("selected videos (raw):", vid_names)

if USE_MOTION and not vid_names and not DRY_RUN:
    raise SystemExit("USE_MOTION=True なのに動画がありません。VIDEO_FILES と input/ を確認してください。")

if vid_names and STRIP_VIDEO_IDENTITY and not DRY_RUN:
    print("STRIP_VIDEO_IDENTITY=True mode=", STRIP_MODE, "— optional identity strip (can weaken motion)")
    vid_names = prepare_motion_refs(INP, vid_names, enabled=True, mode=STRIP_MODE)
    print("selected videos (proxies):", vid_names)
elif vid_names and STRIP_VIDEO_IDENTITY and DRY_RUN:
    print("DRY_RUN: would strip", vid_names, "to", STRIP_MODE)

print(f"length frames={frames(DURATION_S)} (~{DURATION_S}s @24fps)")
print(f"REF_IMAGE_SIZE={REF_IMAGE_SIZE} STRIP={STRIP_VIDEO_IDENTITY}/{STRIP_MODE}")

if not img_names:
    raise SystemExit("人物画像がありません。IMAGE_FILES を指定してください。")

PROMPT = finalize_prompt(PROMPT, img_names, vid_names, DURATION_S, inject_role_lock=INJECT_ROLE_LOCK)
if vid_names and STRIP_VIDEO_IDENTITY:
    PROMPT = (
        "ANTI-LEAK: The motion reference has been identity-stripped (edges/blur). "
        "It contains NO usable face identity. You MUST invent ZERO faces from the video. "
        "All faces MUST come from <Picture 1>"
        + (" and <Picture 2>" if len(img_names) > 1 else "")
        + " only.\n\n"
        + PROMPT
    )
print("\n--- PROMPT (head) ---\n", PROMPT[:2000], ("..." if len(PROMPT) > 2000 else ""))

diff = list((COMFY_DIR / "models/diffusion_models").glob("*ref2va*")) if (COMFY_DIR / "models/diffusion_models").exists() else []
if not diff and not DRY_RUN:
    raise SystemExit("ref2va モデルがありません。セル3 の MODE を both か r2v にするか、セル5 を実行してください。")
unet = diff[0].name if diff else "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
lora_dir = COMFY_DIR / "models/loras"
loras_all = list(lora_dir.glob("*.safetensors")) if lora_dir.exists() else []
lora_name = prefer_ref2v_lora(loras_all, USE_LORA)
if USE_LORA and lora_name and "ref2v" not in lora_name.lower():
    print("WARN: Ref2V turbo が無いので", lora_name, "を使います（FL2V turbo は R2V に不適）")
print("unet", unet, "lora", lora_name)

obj = {}
if not DRY_RUN:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=60) as r:
        obj = json.loads(r.read().decode())
    for need in ["UNETLoader", "CLIPLoader", "VAELoader", "MiniMaxH3ReferenceToVideo", "SaveVideo"]:
        if need not in obj:
            raise SystemExit(f"Missing node {need}. ComfyUI を更新するかセル2を再実行してください。")
    if vid_names and "VHS_LoadVideo" not in obj and "LoadVideo" not in obj:
        raise SystemExit("Video loader missing。セル2 の Video Helper Suite 導入を確認してください。")

has_vhs = ("VHS_LoadVideo" in obj) or DRY_RUN
has_lora_loader = ("LoraLoaderModelOnly" in obj) or DRY_RUN
has_audio = ("VAEDecodeAudio" in obj) or DRY_RUN

graph = build_r2v_graph(
    img_names=img_names,
    vid_names=vid_names if USE_MOTION else [],
    prompt=PROMPT,
    unet=unet,
    lora_name=lora_name,
    lora_strength=float(LORA_STRENGTH),
    width=int(WIDTH),
    height=int(HEIGHT),
    duration_s=float(DURATION_S),
    seed=int(SEED),
    steps=int(STEPS),
    filename_prefix=FILENAME_PREFIX,
    ref_image_size=REF_IMAGE_SIZE,
    use_videos=bool(vid_names and USE_MOTION),
    has_vhs=has_vhs,
    has_lora_loader=has_lora_loader,
    has_audio_decode=has_audio,
    object_info=obj,
)
r_in = graph["20"]["inputs"]
print("  ref_image_size:", r_in.get("ref_image_size"))
for i in range(len(img_names)):
    print(f"  wired image[{i}] {comfy_media_name(img_names[i])} -> {image_ref_key(i)}")
if vid_names and USE_MOTION:
    for vi in range(len(vid_names)):
        print(f"  wired video[{vi}] {comfy_media_name(vid_names[vi])} -> {video_ref_key(vi)}")
errs = assert_graph_identity_motion(
    graph,
    expect_images=len(img_names),
    expect_videos=len(vid_names) if (vid_names and USE_MOTION) else 0,
    prompt=PROMPT,
)
if errs:
    raise SystemExit("GRAPH CHECK FAILED (motion/identity wiring):\n- " + "\n- ".join(errs))

graph_path = Path("/content/h3_r2v_flex_last_graph.json")
if not graph_path.parent.exists():
    graph_path = Path.cwd() / "h3_r2v_flex_last_graph.json"
graph_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")
print("graph:", graph_path)

if DRY_RUN:
    print("DRY_RUN: 投稿しません")
    raise SystemExit(0)

def post_prompt(g):
    body = {"prompt": g, "client_id": str(uuid.uuid4())}
    data = json.dumps(body).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode()), None
    except urllib.error.HTTPError as e:
        return None, f"HTTP {e.code}: {e.read().decode('utf-8', errors='replace')[:4000]}"

def wait_prompt(pid, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{pid}", timeout=60) as r:
            hist = json.loads(r.read().decode())
        entry = hist.get(pid) or {}
        status = entry.get("status") or {}
        if status.get("completed") or entry.get("outputs"):
            return True, entry
        for m in status.get("messages") or []:
            if isinstance(m, list) and m and m[0] == "execution_error":
                return False, m
        time.sleep(5)
    return False, "timeout"

res, last_err = post_prompt(graph)
if not (res and "prompt_id" in res):
    raise SystemExit(f"R2V rejected (動画なしへフォールバックしません): {last_err}")
pid = res["prompt_id"]
print("ACCEPTED", pid)
ok, payload = wait_prompt(pid)
if not ok:
    raise SystemExit(f"R2V runtime fail (動画なしへフォールバックしません): {payload}")
print("DONE", json.dumps((payload or {}).get("outputs"), ensure_ascii=False)[:800])

cands = []
for root in [COMFY_DIR / "output", Path(DRIVE_ROOT) / "output"]:
    if root.exists():
        cands.extend(root.rglob("*.mp4"))
cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
print("\n最新出力:")
for p in cands[:10]:
    print(" ", p, f"{p.stat().st_size/1e6:.2f}MB")
print("unet", unet, "lora", lora_name)


## セル8 の使い方（R2V）

| やりたいこと | 設定 |
|---|---|
| 人物は参照画像 | `IMAGE_FILES` + `REF_IMAGE_SIZE=max` + ROLE LOCK（自動） |
| 動きは参照動画 | `USE_MOTION=True` + `VIDEO_FILES`。失敗しても動画なしへ落とさない |
| 顔が動画側に混ざる | まず画像を増やす（正面・全身）。それでもダメなら `STRIP_VIDEO_IDENTITY=True` |
| 速い生成 | **Ref2V turbo**（FL2V turbo ではない）、`STEPS=4` |

素材:

```
マイドライブ/minimax-h3-comfyui/input/
マイドライブ/minimax-h3-comfyui/models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors
マイドライブ/minimax-h3-comfyui/models/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors
```
